# 09 - Multi-Target ChEMBL Exploration

This notebook explores ChEMBL drug-target evidence for multiple therapeutic targets:

- EGFR
- HER2 / ERBB2
- BRAF
- ALK
- KRAS
- VEGFA
- MET
- PIK3CA

The goal is to move from the original EGFR-only dataset to a reusable multi-target data-preparation process.

Important fixes in this version:

1. Canonical ChEMBL target IDs are used so ChEMBL search does not accidentally select complexes, protein families, non-human targets, or unrelated matches.
2. The ChEMBL mechanism endpoint is read with the correct JSON key: `mechanisms`.
3. The notebook writes both raw snapshots and processed CSV outputs.

## 1. Setup

In [8]:
from pathlib import Path
import json
import re
import time

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "chembl"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw ChEMBL folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Raw ChEMBL folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/chembl
Processed folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


## 2. Canonical Target List

ChEMBL search results can include protein complexes, protein families, mouse targets, or unrelated targets. For this project, we use canonical human ChEMBL target IDs as the final selected targets.

HER2 is represented as `ERBB2` in most biomedical databases, so the canonical symbol is `ERBB2` and the display name is `HER2`.

In [9]:
TARGETS = [
    {
        "target_symbol": "EGFR",
        "target_display_name": "EGFR",
        "target_full_name": "Epidermal growth factor receptor",
        "preferred_chembl_id": "CHEMBL203",
        "aliases": ["ERBB1", "Epidermal growth factor receptor"],
    },
    {
        "target_symbol": "ERBB2",
        "target_display_name": "HER2",
        "target_full_name": "Receptor tyrosine-protein kinase erbB-2",
        "preferred_chembl_id": "CHEMBL1824",
        "aliases": ["HER2", "Receptor tyrosine-protein kinase erbB-2"],
    },
    {
        "target_symbol": "BRAF",
        "target_display_name": "BRAF",
        "target_full_name": "Serine/threonine-protein kinase B-raf",
        "preferred_chembl_id": "CHEMBL5145",
        "aliases": ["B-Raf proto-oncogene", "B-Raf"],
    },
    {
        "target_symbol": "ALK",
        "target_display_name": "ALK",
        "target_full_name": "ALK tyrosine kinase receptor",
        "preferred_chembl_id": "CHEMBL4247",
        "aliases": ["Anaplastic lymphoma kinase"],
    },
    {
        "target_symbol": "KRAS",
        "target_display_name": "KRAS",
        "target_full_name": "GTPase KRas",
        "preferred_chembl_id": "CHEMBL2189121",
        "aliases": ["KRAS proto-oncogene", "K-Ras"],
    },
    {
        "target_symbol": "VEGFA",
        "target_display_name": "VEGFA",
        "target_full_name": "Vascular endothelial growth factor A",
        "preferred_chembl_id": "CHEMBL1783",
        "aliases": ["VEGF", "Vascular endothelial growth factor A"],
    },
    {
        "target_symbol": "MET",
        "target_display_name": "MET",
        "target_full_name": "Hepatocyte growth factor receptor",
        "preferred_chembl_id": "CHEMBL3717",
        "aliases": ["HGFR", "Hepatocyte growth factor receptor"],
    },
    {
        "target_symbol": "PIK3CA",
        "target_display_name": "PIK3CA",
        "target_full_name": "Phosphatidylinositol 4,5-bisphosphate 3-kinase catalytic subunit alpha isoform",
        "preferred_chembl_id": "CHEMBL4005",
        "aliases": [
            "PI3K alpha",
            "Phosphatidylinositol-4,5-bisphosphate 3-kinase catalytic subunit alpha",
        ],
    },
]

targets_df = pd.DataFrame(TARGETS)
display(targets_df)

,target_symbol,target_display_name,target_full_name,preferred_chembl_id,aliases
0,EGFR,EGFR,Epidermal growth factor receptor,CHEMBL203,"[ERBB1, Epidermal growth factor receptor]"
1,ERBB2,HER2,Receptor tyrosine-protein kinase erbB-2,CHEMBL1824,"[HER2, Receptor tyrosine-protein kinase erbB-2]"
2,BRAF,BRAF,Serine/threonine-protein kinase B-raf,CHEMBL5145,"[B-Raf proto-oncogene, B-Raf]"
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,[Anaplastic lymphoma kinase]
4,KRAS,KRAS,GTPase KRas,CHEMBL2189121,"[KRAS proto-oncogene, K-Ras]"
5,VEGFA,VEGFA,Vascular endothelial growth factor A,CHEMBL1783,"[VEGF, Vascular endothelial growth factor A]"
6,MET,MET,Hepatocyte growth factor receptor,CHEMBL3717,"[HGFR, Hepatocyte growth factor receptor]"
7,PIK3CA,PIK3CA,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",CHEMBL4005,"[PI3K alpha, Phosphatidylinositol-4,5-bisphosp..."


## 3. Helper Functions

In [10]:
CHEMBL_BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"


def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def normalize_name(value) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


def save_json(path: Path, data) -> None:
    with path.open("w") as f:
        json.dump(data, f, indent=2)


def chembl_get(url, params=None, retries=4, pause=3):
    params = params or {}

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=(10, 120))
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as error:
            print(f"Attempt {attempt}/{retries} failed: {error}")
            if attempt == retries:
                raise
            time.sleep(pause * attempt)


def chembl_get_all(resource: str, records_key: str, params=None, limit=1000):
    """Fetch all paginated records from a ChEMBL endpoint.

    Example:
    - resource='mechanism'
    - records_key='mechanisms'

    The endpoint name can be singular while the JSON key is plural, so the
    records key must be explicit.
    """
    params = dict(params or {})
    params["limit"] = limit
    params["offset"] = 0

    all_records = []
    url = f"{CHEMBL_BASE_URL}/{resource}.json"

    while True:
        data = chembl_get(url, params=params)
        records = data.get(records_key, [])
        all_records.extend(records)

        page_meta = data.get("page_meta", {})
        if not page_meta.get("next"):
            break

        params["offset"] += limit

    return all_records


def approval_label(max_phase):
    try:
        phase = float(max_phase)
    except (TypeError, ValueError):
        return "Research / Unknown"

    if phase >= 4:
        return "Approved"
    if phase >= 3:
        return "Investigational (Phase 3)"
    if phase >= 2:
        return "Investigational (Phase 2)"
    if phase >= 1:
        return "Investigational (Phase 1)"
    return "Research / Unknown"

## 4. Resolve Target Records

This step saves ChEMBL search results for transparency, then fetches the canonical target record using `preferred_chembl_id`.

In [11]:
target_rows = []
target_candidate_rows = []

for target in TARGETS:
    target_symbol = target["target_symbol"]
    target_display_name = target["target_display_name"]
    preferred_chembl_id = target["preferred_chembl_id"]

    print(f"\nResolving target: {target_symbol} / {target_display_name}")

    search_terms = [target_symbol, target_display_name] + target.get("aliases", [])
    seen_candidates = {}

    for term in search_terms:
        search_url = f"{CHEMBL_BASE_URL}/target/search.json"
        search_data = chembl_get(search_url, params={"q": term})

        raw_search_file = RAW_DIR / f"{slug(target_symbol)}_target_search_{slug(term)}_raw.json"
        save_json(raw_search_file, search_data)

        for candidate in search_data.get("targets", []):
            candidate_id = candidate.get("target_chembl_id")
            if candidate_id:
                seen_candidates[candidate_id] = candidate

    for rank, candidate in enumerate(seen_candidates.values(), start=1):
        target_candidate_rows.append(
            {
                "target_symbol": target_symbol,
                "target_display_name": target_display_name,
                "rank_from_search": rank,
                "candidate_target_chembl_id": candidate.get("target_chembl_id"),
                "candidate_pref_name": candidate.get("pref_name"),
                "candidate_organism": candidate.get("organism"),
                "candidate_target_type": candidate.get("target_type"),
                "is_preferred_target": candidate.get("target_chembl_id") == preferred_chembl_id,
            }
        )

    target_url = f"{CHEMBL_BASE_URL}/target/{preferred_chembl_id}.json"
    target_record = chembl_get(target_url)

    raw_target_file = RAW_DIR / f"{slug(target_symbol)}_target_record_raw.json"
    save_json(raw_target_file, target_record)

    selected_id = target_record.get("target_chembl_id")
    resolution_status = "resolved" if selected_id == preferred_chembl_id else "needs_review"

    target_rows.append(
        {
            "target_symbol": target_symbol,
            "target_display_name": target_display_name,
            "target_full_name": target["target_full_name"],
            "target_chembl_id": selected_id,
            "target_pref_name": target_record.get("pref_name"),
            "target_organism": target_record.get("organism"),
            "target_type": target_record.get("target_type"),
            "resolution_status": resolution_status,
        }
    )

    print(
        "  Selected:",
        selected_id,
        "-",
        target_record.get("pref_name"),
        "|",
        target_record.get("organism"),
        "|",
        target_record.get("target_type"),
    )

multi_target_chembl_targets_df = pd.DataFrame(target_rows)
multi_target_chembl_target_candidates_df = pd.DataFrame(target_candidate_rows)

targets_file = PROCESSED_DIR / "multi_target_chembl_targets.csv"
target_candidates_file = PROCESSED_DIR / "multi_target_chembl_target_candidates.csv"

multi_target_chembl_targets_df.to_csv(targets_file, index=False)
multi_target_chembl_target_candidates_df.to_csv(target_candidates_file, index=False)

print("\nSaved:", targets_file)
print("Saved:", target_candidates_file)

display(multi_target_chembl_targets_df)


Resolving target: EGFR / EGFR
  Selected: CHEMBL203 - Epidermal growth factor receptor | Homo sapiens | SINGLE PROTEIN

Resolving target: ERBB2 / HER2
  Selected: CHEMBL1824 - Receptor tyrosine-protein kinase erbB-2 | Homo sapiens | SINGLE PROTEIN

Resolving target: BRAF / BRAF
  Selected: CHEMBL5145 - Serine/threonine-protein kinase B-raf | Homo sapiens | SINGLE PROTEIN

Resolving target: ALK / ALK
  Selected: CHEMBL4247 - ALK tyrosine kinase receptor | Homo sapiens | SINGLE PROTEIN

Resolving target: KRAS / KRAS
  Selected: CHEMBL2189121 - GTPase KRas | Homo sapiens | SINGLE PROTEIN

Resolving target: VEGFA / VEGFA
  Selected: CHEMBL1783 - Vascular endothelial growth factor A, long form | Homo sapiens | SINGLE PROTEIN

Resolving target: MET / MET
  Selected: CHEMBL3717 - Hepatocyte growth factor receptor | Homo sapiens | SINGLE PROTEIN

Resolving target: PIK3CA / PIK3CA
  Selected: CHEMBL4005 - Phosphatidylinositol 4,5-bisphosphate 3-kinase catalytic subunit alpha isoform | Homo sap

,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,target_organism,target_type,resolution_status
0,EGFR,EGFR,Epidermal growth factor receptor,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN,resolved
1,ERBB2,HER2,Receptor tyrosine-protein kinase erbB-2,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,SINGLE PROTEIN,resolved
2,BRAF,BRAF,Serine/threonine-protein kinase B-raf,CHEMBL5145,Serine/threonine-protein kinase B-raf,Homo sapiens,SINGLE PROTEIN,resolved
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,resolved
4,KRAS,KRAS,GTPase KRas,CHEMBL2189121,GTPase KRas,Homo sapiens,SINGLE PROTEIN,resolved
5,VEGFA,VEGFA,Vascular endothelial growth factor A,CHEMBL1783,"Vascular endothelial growth factor A, long form",Homo sapiens,SINGLE PROTEIN,resolved
6,MET,MET,Hepatocyte growth factor receptor,CHEMBL3717,Hepatocyte growth factor receptor,Homo sapiens,SINGLE PROTEIN,resolved
7,PIK3CA,PIK3CA,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",CHEMBL4005,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",Homo sapiens,SINGLE PROTEIN,resolved


## 5. Fetch Mechanisms And Molecule Metadata

In [12]:
molecule_cache = {}


def get_molecule_info(molecule_chembl_id):
    if not molecule_chembl_id:
        return {}

    if molecule_chembl_id in molecule_cache:
        return molecule_cache[molecule_chembl_id]

    molecule_url = f"{CHEMBL_BASE_URL}/molecule/{molecule_chembl_id}.json"

    try:
        molecule_data = chembl_get(molecule_url)
    except Exception as error:
        print("Could not fetch molecule:", molecule_chembl_id, error)
        molecule_data = {}

    molecule_cache[molecule_chembl_id] = molecule_data
    return molecule_data


mechanism_rows = []

for _, target in multi_target_chembl_targets_df.iterrows():
    target_symbol = target["target_symbol"]
    target_display_name = target["target_display_name"]
    target_chembl_id = target["target_chembl_id"]

    print(f"\nFetching mechanisms for {target_symbol} ({target_chembl_id})")

    mechanisms = chembl_get_all(
        resource="mechanism",
        records_key="mechanisms",
        params={"target_chembl_id": target_chembl_id},
        limit=1000,
    )

    raw_mechanisms_file = RAW_DIR / f"{slug(target_symbol)}_chembl_mechanisms_raw.json"
    save_json(raw_mechanisms_file, mechanisms)

    print("  Mechanism records:", len(mechanisms))

    for mechanism in mechanisms:
        molecule_chembl_id = mechanism.get("molecule_chembl_id")
        molecule = get_molecule_info(molecule_chembl_id)

        drug_name = (
            molecule.get("pref_name")
            or mechanism.get("molecule_pref_name")
            or molecule_chembl_id
        )

        max_phase = molecule.get("max_phase")
        molecule_synonyms = molecule.get("molecule_synonyms") or []
        synonym_values = [
            item.get("molecule_synonym")
            for item in molecule_synonyms
            if item.get("molecule_synonym")
        ]

        mechanism_rows.append(
            {
                "target_symbol": target_symbol,
                "target_display_name": target_display_name,
                "target_full_name": target["target_full_name"],
                "target_chembl_id": target_chembl_id,
                "target_pref_name": target["target_pref_name"],
                "target_organism": target["target_organism"],
                "target_type": target["target_type"],
                "molecule_chembl_id": molecule_chembl_id,
                "drug_name": drug_name,
                "normalised_drug_name": normalize_name(drug_name),
                "molecule_type": molecule.get("molecule_type"),
                "first_approval": molecule.get("first_approval"),
                "max_phase": max_phase,
                "approval_status": approval_label(max_phase),
                "action_type": mechanism.get("action_type"),
                "mechanism_of_action": mechanism.get("mechanism_of_action"),
                "mechanism_comment": mechanism.get("mechanism_comment"),
                "selectivity_comment": mechanism.get("selectivity_comment"),
                "binding_site_comment": mechanism.get("binding_site_comment"),
                "mechanism_refs": json.dumps(mechanism.get("mechanism_refs", [])),
                "molecule_synonyms": " | ".join(sorted(set(synonym_values))[:20]),
            }
        )

multi_target_chembl_mechanisms_df = pd.DataFrame(mechanism_rows)

if not multi_target_chembl_mechanisms_df.empty:
    multi_target_chembl_mechanisms_df = (
        multi_target_chembl_mechanisms_df
        .drop_duplicates(
            subset=[
                "target_symbol",
                "molecule_chembl_id",
                "mechanism_of_action",
                "action_type",
            ]
        )
        .sort_values(["target_symbol", "drug_name"])
        .reset_index(drop=True)
    )

mechanisms_file = PROCESSED_DIR / "multi_target_chembl_mechanisms.csv"
multi_target_chembl_mechanisms_df.to_csv(mechanisms_file, index=False)

print("\nSaved:", mechanisms_file)
print("Rows:", len(multi_target_chembl_mechanisms_df))

display(multi_target_chembl_mechanisms_df.head(20))


Fetching mechanisms for EGFR (CHEMBL203)
  Mechanism records: 90

Fetching mechanisms for ERBB2 (CHEMBL1824)
  Mechanism records: 41

Fetching mechanisms for BRAF (CHEMBL5145)
  Mechanism records: 14

Fetching mechanisms for ALK (CHEMBL4247)
  Mechanism records: 11

Fetching mechanisms for KRAS (CHEMBL2189121)
  Mechanism records: 2

Fetching mechanisms for VEGFA (CHEMBL1783)
  Mechanism records: 13

Fetching mechanisms for MET (CHEMBL3717)
  Mechanism records: 39

Fetching mechanisms for PIK3CA (CHEMBL4005)
  Mechanism records: 9

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_chembl_mechanisms.csv
Rows: 207


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,target_organism,target_type,molecule_chembl_id,drug_name,normalised_drug_name,...,first_approval,max_phase,approval_status,action_type,mechanism_of_action,mechanism_comment,selectivity_comment,binding_site_comment,mechanism_refs,molecule_synonyms
0,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL3707320,ALECTINIB HYDROCHLORIDE,ALECTINIBHYDROCHLORIDE,...,2015.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""label/2015/208434s000lbl.pdf"", ""r...",AF-802 HYDROCHLORIDE | Alecensa | Alectinib hy...
1,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL3545360,ASP-3026,ASP3026,...,NaN,1.0,Investigational (Phase 1),INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""24419060"", ""ref_type"": ""PubMed"", ...",ASP3026 | Asp 3026 | Asp-3026
2,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL3545311,BRIGATINIB,BRIGATINIB,...,2017.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Inhibits ALK and EGFR gatekeeper mutants. Does...,NaN,NaN,"[{""ref_id"": ""http://www.ariad.com/AP26113"", ""r...",ALUNBRIG | AP 26113 | AP-26113 | AP26113 | Alu...
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL3951811,CEP-37440,CEP37440,...,NaN,1.0,Investigational (Phase 1),INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""10.1158/1538-7445.AM2015-3232"", ""...",Alk-fak inhibitor cep-37440 | Cep-37440
4,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL2403108,CERITINIB,CERITINIB,...,2014.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""label/2014/205755s000lbl.pdf"", ""r...",4MK | CERITINIB | CERITINIB (LDK378) | CERITIN...
5,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL3899477,CONTELTINIB,CONTELTINIB,...,NaN,1.0,Investigational (Phase 1),INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""NCT02695550"", ""ref_type"": ""Clinic...",Conteltinib
6,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL601719,CRIZOTINIB,CRIZOTINIB,...,2011.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""label/2012/202570s003lbl.pdf"", ""r...",Crizotinib | NSC-756645 | PF-02341066 | Pf-234...
7,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL4113131,ENSARTINIB,ENSARTINIB,...,2024.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,NaN,NaN,NaN,"[{""ref_id"": ""http://www.xcovery.com/our-scienc...",Ensacove | Ensartinib | X-396
8,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL1983268,ENTRECTINIB,ENTRECTINIB,...,2019.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,"TRK, ROS1, or ALK kinase fusion proteins can c...","Inhibits the TRK, ROS1 and ALK kinases at 0.1 ...",NaN,"[{""ref_id"": ""label/2019/212725s000lbl.pdf"", ""r...",Entrectinib | NMS-E628 | RXDX-101 | Rozlytrek ...
9,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,CHEMBL3286830,LORLATINIB,LORLATINIB,...,2018.0,4.0,Approved,INHIBITOR,ALK tyrosine kinase receptor inhibitor,"Active against ALK-driven tumors, inhibits ALK...","Active against other kinases: ROS1, TYK1, FER,...",NaN,"[{""ref_id"": ""label/2018/210868s000lbl.pdf"", ""r...",Lorbrena | Lorlatinib | Lorviqua | PF-06463922


## 6. Build Multi-Target Drug Recommendations

In [13]:
recommendation_columns = [
    "target_symbol",
    "target_display_name",
    "target_full_name",
    "target_chembl_id",
    "target_pref_name",
    "drug_name",
    "molecule_chembl_id",
    "molecule_type",
    "action_type",
    "mechanism_of_action",
    "approval_status",
    "max_phase",
    "first_approval",
]

if multi_target_chembl_mechanisms_df.empty:
    multi_target_drug_recommendations_df = pd.DataFrame(columns=recommendation_columns)
else:
    multi_target_drug_recommendations_df = multi_target_chembl_mechanisms_df[
        recommendation_columns
    ].copy()

    multi_target_drug_recommendations_df = (
        multi_target_drug_recommendations_df
        .drop_duplicates(
            subset=[
                "target_symbol",
                "drug_name",
                "molecule_chembl_id",
                "mechanism_of_action",
                "action_type",
            ]
        )
        .sort_values(["target_symbol", "drug_name"])
        .reset_index(drop=True)
    )

recommendations_file = PROCESSED_DIR / "multi_target_drug_recommendations.csv"
multi_target_drug_recommendations_df.to_csv(recommendations_file, index=False)

print("Saved:", recommendations_file)
print("Rows:", len(multi_target_drug_recommendations_df))

display(multi_target_drug_recommendations_df.head(30))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_drug_recommendations.csv
Rows: 207


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,drug_name,molecule_chembl_id,molecule_type,action_type,mechanism_of_action,approval_status,max_phase,first_approval
0,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2015.0
1,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ASP-3026,CHEMBL3545360,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
2,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2017.0
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CEP-37440,CHEMBL3951811,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
4,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CERITINIB,CHEMBL2403108,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2014.0
5,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CONTELTINIB,CHEMBL3899477,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
6,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CRIZOTINIB,CHEMBL601719,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2011.0
7,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ENSARTINIB,CHEMBL4113131,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2024.0
8,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ENTRECTINIB,CHEMBL1983268,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2019.0
9,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,LORLATINIB,CHEMBL3286830,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2018.0


## 7. Save Optional Per-Target Recommendation Files

In [14]:
per_target_files = []

for target_symbol, target_df in multi_target_drug_recommendations_df.groupby("target_symbol"):
    per_target_file = PROCESSED_DIR / f"{slug(target_symbol)}_drug_recommendations.csv"
    target_df.to_csv(per_target_file, index=False)
    per_target_files.append(per_target_file)

print("Per-target files:")
for path in per_target_files:
    print("-", path)

Per-target files:
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/alk_drug_recommendations.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/braf_drug_recommendations.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_drug_recommendations.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/erbb2_drug_recommendations.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/kras_drug_recommendations.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy

## 8. Coverage Summary

In [15]:
coverage_rows = []

for target in TARGETS:
    target_symbol = target["target_symbol"]

    target_record = multi_target_chembl_targets_df[
        multi_target_chembl_targets_df["target_symbol"] == target_symbol
    ]

    target_drugs = multi_target_drug_recommendations_df[
        multi_target_drug_recommendations_df["target_symbol"] == target_symbol
    ]

    if target_record.empty:
        resolution_status = "not_found"
        target_chembl_id = None
        target_pref_name = None
        target_type = None
    else:
        row = target_record.iloc[0]
        resolution_status = row["resolution_status"]
        target_chembl_id = row["target_chembl_id"]
        target_pref_name = row["target_pref_name"]
        target_type = row["target_type"]

    approved_count = (
        int(target_drugs["approval_status"].eq("Approved").sum())
        if not target_drugs.empty
        else 0
    )

    coverage_rows.append(
        {
            "target_symbol": target_symbol,
            "target_display_name": target["target_display_name"],
            "source": "ChEMBL",
            "target_resolution_status": resolution_status,
            "target_chembl_id": target_chembl_id,
            "target_pref_name": target_pref_name,
            "target_type": target_type,
            "raw_saved": True,
            "processed_saved": True,
            "drug_record_count": len(target_drugs),
            "approved_drug_count": approved_count,
            "status": "working" if len(target_drugs) > 0 else "needs_review",
            "notes": "" if len(target_drugs) > 0 else "No ChEMBL mechanism rows found for this canonical target.",
        }
    )

multi_target_chembl_coverage_df = pd.DataFrame(coverage_rows)

coverage_file = PROCESSED_DIR / "multi_target_chembl_coverage_summary.csv"
multi_target_chembl_coverage_df.to_csv(coverage_file, index=False)

print("Saved:", coverage_file)
display(multi_target_chembl_coverage_df)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_chembl_coverage_summary.csv


,target_symbol,target_display_name,source,target_resolution_status,target_chembl_id,target_pref_name,target_type,raw_saved,processed_saved,drug_record_count,approved_drug_count,status,notes
0,EGFR,EGFR,ChEMBL,resolved,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN,True,True,79,20,working,
1,ERBB2,HER2,ChEMBL,resolved,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,SINGLE PROTEIN,True,True,40,14,working,
2,BRAF,BRAF,ChEMBL,resolved,CHEMBL5145,Serine/threonine-protein kinase B-raf,SINGLE PROTEIN,True,True,14,5,working,
3,ALK,ALK,ChEMBL,resolved,CHEMBL4247,ALK tyrosine kinase receptor,SINGLE PROTEIN,True,True,11,7,working,
4,KRAS,KRAS,ChEMBL,resolved,CHEMBL2189121,GTPase KRas,SINGLE PROTEIN,True,True,2,2,working,
5,VEGFA,VEGFA,ChEMBL,resolved,CHEMBL1783,"Vascular endothelial growth factor A, long form",SINGLE PROTEIN,True,True,13,7,working,
6,MET,MET,ChEMBL,resolved,CHEMBL3717,Hepatocyte growth factor receptor,SINGLE PROTEIN,True,True,39,8,working,
7,PIK3CA,PIK3CA,ChEMBL,resolved,CHEMBL4005,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",SINGLE PROTEIN,True,True,9,4,working,


## 9. Final Summary

In [16]:
print("Multi-Target ChEMBL Exploration Complete")
print("=" * 80)

print("\nTargets resolved:")
print(multi_target_chembl_targets_df["resolution_status"].value_counts(dropna=False).to_string())

print("\nSelected ChEMBL targets:")
print(
    multi_target_chembl_targets_df[
        [
            "target_symbol",
            "target_display_name",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "target_type",
        ]
    ].to_string(index=False)
)

print("\nDrug recommendation rows by target:")
if not multi_target_drug_recommendations_df.empty:
    print(
        multi_target_drug_recommendations_df
        .groupby("target_symbol")
        .size()
        .sort_values(ascending=False)
        .to_string()
    )
else:
    print("No drug recommendation rows found.")

print("\nApproval status distribution:")
if not multi_target_drug_recommendations_df.empty:
    print(multi_target_drug_recommendations_df["approval_status"].value_counts().to_string())
else:
    print("No approval status values found.")

print("\nFiles created:")
for path in [
    targets_file,
    target_candidates_file,
    mechanisms_file,
    recommendations_file,
    coverage_file,
]:
    print("-", path)

for path in per_target_files:
    print("-", path)

display(multi_target_chembl_coverage_df)

Multi-Target ChEMBL Exploration Complete

Targets resolved:
resolution_status
resolved    8

Selected ChEMBL targets:
target_symbol target_display_name target_chembl_id                                                               target_pref_name target_organism    target_type
         EGFR                EGFR        CHEMBL203                                               Epidermal growth factor receptor    Homo sapiens SINGLE PROTEIN
        ERBB2                HER2       CHEMBL1824                                        Receptor tyrosine-protein kinase erbB-2    Homo sapiens SINGLE PROTEIN
         BRAF                BRAF       CHEMBL5145                                          Serine/threonine-protein kinase B-raf    Homo sapiens SINGLE PROTEIN
          ALK                 ALK       CHEMBL4247                                                   ALK tyrosine kinase receptor    Homo sapiens SINGLE PROTEIN
         KRAS                KRAS    CHEMBL2189121                           

,target_symbol,target_display_name,source,target_resolution_status,target_chembl_id,target_pref_name,target_type,raw_saved,processed_saved,drug_record_count,approved_drug_count,status,notes
0,EGFR,EGFR,ChEMBL,resolved,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN,True,True,79,20,working,
1,ERBB2,HER2,ChEMBL,resolved,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,SINGLE PROTEIN,True,True,40,14,working,
2,BRAF,BRAF,ChEMBL,resolved,CHEMBL5145,Serine/threonine-protein kinase B-raf,SINGLE PROTEIN,True,True,14,5,working,
3,ALK,ALK,ChEMBL,resolved,CHEMBL4247,ALK tyrosine kinase receptor,SINGLE PROTEIN,True,True,11,7,working,
4,KRAS,KRAS,ChEMBL,resolved,CHEMBL2189121,GTPase KRas,SINGLE PROTEIN,True,True,2,2,working,
5,VEGFA,VEGFA,ChEMBL,resolved,CHEMBL1783,"Vascular endothelial growth factor A, long form",SINGLE PROTEIN,True,True,13,7,working,
6,MET,MET,ChEMBL,resolved,CHEMBL3717,Hepatocyte growth factor receptor,SINGLE PROTEIN,True,True,39,8,working,
7,PIK3CA,PIK3CA,ChEMBL,resolved,CHEMBL4005,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",SINGLE PROTEIN,True,True,9,4,working,
